In [1]:
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

In [14]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [33]:
#rutas (ALEJANDRO - SSD)
manzanas_gpkg_path = r"D:\AFC\SSD\VOCES\DATA\Tratados\NAC\INT_MZN_NAC\Mzn_Tipologías_NAC_ImpEq_BufferB.gpkg"
capa = 'Mzn_Subtipologias_UeEQ_filtrado'

# Cargar el archivo de manzanas
gdf_manzanas = gpd.read_file(manzanas_gpkg_path, layer=capa)

In [41]:
gdf_manzanas_intacto = gdf_manzanas.copy()

# Nuevo procesamiento

## op 1

In [4]:
gdf_manzanas.columns

Index(['CVEGEO', 'POPTOT2010', 'POPTOT2020', 'VIVTOT2010', 'VIVTOT2020',
       'TVIVHAB2010', 'TVIVHAB2020', 'TVIVPAR2010', 'TVIVPAR2020',
       'TVIVPARHAB2010',
       ...
       'CVE_SAL_B', 'CVE_REC_B', 'CVE_SER_B', 'CVE_EQ_B', 'CVE_EDU_C',
       'CVE_SAL_C', 'CVE_REC_C', 'CVE_SER_C', 'CVE_EQ_C', 'geometry'],
      dtype='object', length=115)

In [6]:
for col in ['IMP_EDU_C', 'IMP_SAL_C', 'IMP_REC_C', 'IMP_SER_C']:
    gdf_manzanas[col + '_pct'] = gdf_manzanas[col].rank(pct=True)

In [8]:
gdf_manzanas['ICE'] = (
    gdf_manzanas['IMP_EDU_C_pct'] +
    gdf_manzanas['IMP_SAL_C_pct'] +
    gdf_manzanas['IMP_REC_C_pct'] +
    gdf_manzanas['IMP_SER_C_pct']
) / 4

In [9]:
gdf_manzanas['ICE_cat'] = pd.qcut(gdf_manzanas['ICE'], 6, labels=['Muy baja','Baja','Media','Alta','Muy alta', 'Altísima'])

In [11]:
tabla_revision = (
    gdf_manzanas.groupby(['ICE_cat'])
      .size()
      .reset_index(name='n')
      .sort_values('ICE_cat')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

C:\Users\Alejandro\AppData\Local\Temp\ipykernel_9900\2154407781.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gdf_manzanas.groupby(['ICE_cat'])
C:\Users\Alejandro\AppData\Local\Temp\ipykernel_9900\2154407781.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)


,ICE_cat,n
0,Muy baja,156385.0
1,Baja,156383.0
2,Media,156374.0
3,Alta,156386.0
4,Muy alta,156375.0
5,Altísima,156381.0
Total,NaN,938284.0


## op 2

In [12]:
T = {
    'EDU': 1140,
    'SAL': 13.5,
    'REC': 185.5,
    'SER': 18.5
}

gdf_manzanas['RC_EDU'] = gdf_manzanas['IMP_EDU_C'] / T['EDU']
gdf_manzanas['RC_SAL'] = gdf_manzanas['IMP_SAL_C'] / T['SAL']
gdf_manzanas['RC_REC'] = gdf_manzanas['IMP_REC_C'] / T['REC']
gdf_manzanas['RC_SER'] = gdf_manzanas['IMP_SER_C'] / T['SER']

In [14]:
gdf_manzanas['ICE_norm'] = (
    gdf_manzanas['RC_EDU'] +
    gdf_manzanas['RC_SAL'] +
    gdf_manzanas['RC_REC'] +
    gdf_manzanas['RC_SER']
) / 4

In [18]:
# Mostrar estadísticas descriptivas sin notación exponencial
stats = gdf_manzanas['ICE_norm'].describe()
print(stats)

count    938284.000000
mean          5.351988
std           4.001786
min           0.000000
25%           2.282142
50%           4.355480
75%           7.459351
max          23.245143
Name: ICE_norm, dtype: float64


In [15]:
def clasificar_6(x):
    if x < 0.5:
        return 'F'
    elif x < 0.75:
        return 'E'
    elif x < 1:
        return 'D'
    elif x < 1.5:
        return 'C'
    elif x < 2.5:
        return 'B'
    else:
        return 'A'

gdf_manzanas['ICE_cat'] = gdf_manzanas['ICE_norm'].apply(clasificar_6)

In [17]:
gdf_manzanas['ICE_cat'].value_counts(normalize=True)

ICE_cat
A    0.722267
B    0.130323
C    0.064597
D    0.035244
E    0.027543
F    0.020026
Name: proportion, dtype: float64

## op 3

In [50]:
T = {
    'EDU': gdf_manzanas['IMP_EDU_C'].quantile(0.5),
    'SAL': gdf_manzanas['IMP_SAL_C'].quantile(0.5),
    'REC': gdf_manzanas['IMP_REC_C'].quantile(0.5),
    'SER': gdf_manzanas['IMP_SER_C'].quantile(0.5)
}

In [51]:
gdf_manzanas['IMP_EQ_nueva'] = (
    gdf_manzanas['IMP_EDU_C']/T['EDU'] +
    gdf_manzanas['IMP_SAL_C']/T['SAL'] +
    gdf_manzanas['IMP_REC_C']/T['REC'] +
    gdf_manzanas['IMP_SER_C']/T['SER']
) / 4

In [52]:
# Mostrar estadísticas descriptivas sin notación exponencial
stats = gdf_manzanas['IMP_EQ_nueva'].describe()
print(stats)

count    938284.000000
mean          2.157746
std           1.996988
min           0.000000
25%           0.803879
50%           1.469880
75%           2.968614
max          14.363233
Name: IMP_EQ_nueva, dtype: float64


In [55]:
def clasificar_6(x):
    if x < 0.6:
        return 'F'
    elif x < 0.85:
        return 'E'
    elif x < 1:
        return 'D'
    elif x < 1.6:
        return 'C'
    elif x < 2.8:
        return 'B'
    else:
        return 'A'

gdf_manzanas['CVE_EQ_C_nueva'] = gdf_manzanas['IMP_EQ_nueva'].apply(clasificar_6)

In [56]:
gdf_manzanas['CVE_EQ_C_nueva'].value_counts(normalize=True)

CVE_EQ_C_nueva
A    0.268827
C    0.202672
B    0.197285
F    0.174093
E    0.094085
D    0.063038
Name: proportion, dtype: float64

### revisión 

In [30]:
gdf_manzanas.groupby('CVE_MET')['ICE_norm'].mean()
gdf_manzanas.groupby('CVE_MET')['ICE_norm'].median()
gdf_manzanas.groupby('CVE_MET')['ICE_cat'].value_counts(normalize=True)

CVE_MET  ICE_cat
01.1.01  A          0.860695
         C          0.042405
         D          0.026567
         B          0.025886
         F          0.025290
         E          0.019159
02.1.01  B          0.361841
         C          0.218607
         F          0.200629
         E          0.103462
         D          0.069591
         A          0.045869
02.2.02  F          0.957089
         E          0.030535
         C          0.009685
         D          0.002690
02.2.03  C          0.515580
         F          0.180968
         B          0.115351
         D          0.096440
         E          0.091662
03.2.01  F          0.298453
         C          0.288893
         E          0.206501
         D          0.118025
         B          0.088128
03.2.02  F          0.607504
         C          0.205410
         E          0.103490
         D          0.075567
         B          0.008028
04.2.01  C          0.297948
         E          0.289740
         D          0.1991

In [31]:
df = gdf_manzanas.groupby('CVE_MET')['ICE_cat'].value_counts(normalize=True).reset_index(name='proportion')

In [34]:
df.sort_values(by=['CVE_MET','ICE_cat'])

,CVE_MET,ICE_cat,proportion
0,01.1.01,A,0.860695
3,01.1.01,B,0.025886
1,01.1.01,C,0.042405
2,01.1.01,D,0.026567
5,01.1.01,E,0.019159
4,01.1.01,F,0.025290
11,02.1.01,A,0.045869
6,02.1.01,B,0.361841
7,02.1.01,C,0.218607
10,02.1.01,D,0.069591


In [37]:
gdf_manzanas.groupby('CVE_MET')['ICE_norm'].mean().reset_index(name='%')

,CVE_MET,%
0,01.1.01,4.775755
1,02.1.01,1.404099
2,02.2.02,0.298224
3,02.2.03,1.068654
4,03.2.01,0.854589
5,03.2.02,0.562687
6,04.2.01,0.895645
7,05.1.01,1.227724
8,05.1.02,0.541388
9,05.1.03,1.180157


### Guardado paralelo

In [57]:
list(gdf_manzanas.columns.values)

['CVEGEO',
 'POPTOT2010',
 'POPTOT2020',
 'VIVTOT2010',
 'VIVTOT2020',
 'TVIVHAB2010',
 'TVIVHAB2020',
 'TVIVPAR2010',
 'TVIVPAR2020',
 'TVIVPARHAB2010',
 'TVIVPARHAB2020',
 'VIVPAR_HAB2010',
 'VIVPAR_HAB2020',
 'VIVPAR_UT2010',
 'VIVPAR_UT2020',
 'OCUPVIVPAR2010',
 'OCUPVIVPAR2020',
 'VIVPAR_DES2010',
 'VIVPAR_DES2020',
 'VIV1C2010',
 'VIV1C2020',
 'VIV2C2010',
 'VIV2C2020',
 'VIV3C2010',
 'VIV3C2020',
 'CVE_ENT',
 'CVE_MUN',
 'CVE_LOC',
 'CVE_AGEB',
 'CVE_MZA',
 'AMBITO',
 'TIPOMZA',
 'NOMGEO',
 'index_right',
 'CALIF_CLAS',
 'CVEMUN',
 'CVE_MET',
 'NOM_MET',
 'TIPO_MET',
 'TIPO_MUN',
 'area_ha',
 'DESHABITACION',
 'DIFPOP',
 'DIFVIV',
 'RELPOP',
 'RELVIV',
 'CVEPOP',
 'CVEVIV',
 'CVE',
 'TIPOLOGIA',
 'DENS_POP10',
 'DENS_POP20',
 'DENS_VIV10',
 'DENS_VIV20',
 'CUARTOS2010',
 'CUARTOS2020',
 'CPERC2010',
 'CPERC2020',
 'HABXVIV2010',
 'HABXVIV2020',
 'R_DESHAB',
 'SUBTIPOLOGÍA',
 'UE_10',
 'Emp_10',
 'UE_20',
 'Emp_20',
 'TAMAÑO',
 'TASA_EMP_MUN',
 'DIF_EMP',
 'DIF_UEC',
 'REL_EMP',


In [58]:
manzanas_gpkg_path = r"D:\AFC\SSD\VOCES\DATA\Tratados\NAC\INT_MZN_NAC\Mzn_Tipologías_NAC_ImpEq_BufferB_rev.gpkg"
capa = 'Mzn_Subtipologias_UeEQ_filtrado_rev1'

#Guardar la capa resultante en un archivo .gpkg
gdf_manzanas.to_file(manzanas_gpkg_path, layer=capa, driver="GPKG")

# Arreglo metodología vieja

In [35]:
cols = ['CVE_EDU_C', 'CVE_SAL_C', 'CVE_REC_C', 'CVE_SER_C']

# Conteos por categoría
gdf_manzanas['count_S'] = (gdf_manzanas[cols] == 'S').sum(axis=1)
gdf_manzanas['count_C'] = (gdf_manzanas[cols] == 'C').sum(axis=1)
gdf_manzanas['count_D'] = (gdf_manzanas[cols] == 'D').sum(axis=1)

In [36]:
gdf_manzanas[['count_S','count_C','count_D']].sum(axis=1).value_counts()

4    938284
Name: count, dtype: int64

In [37]:
gdf_manzanas['combo_signature'] = (
    'S' + gdf_manzanas['count_S'].astype(str) + 
    '_C' + gdf_manzanas['count_C'].astype(str) + 
    '_D' + gdf_manzanas['count_D'].astype(str)
)

In [38]:
conteo_combinaciones = (
    gdf_manzanas['combo_signature']
    .value_counts()
    .reset_index()
)

conteo_combinaciones.columns = ['combinacion', 'frecuencia']
conteo_combinaciones

,combinacion,frecuencia
0,S3_C0_D1,281258
1,S2_C0_D2,184502
2,S2_C1_D1,121129
3,S3_C1_D0,96072
4,S1_C1_D2,73222
5,S1_C0_D3,49978
6,S4_C0_D0,30397
7,S0_C1_D3,28409
8,S1_C2_D1,22708
9,S0_C0_D4,18708


In [39]:
tabla_revision = (
    gdf_manzanas.groupby(['CVE_EQ_C'])
      .size()
      .reset_index(name='n')
      .sort_values('CVE_EQ_C')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,CVE_EQ_C,n
0,A,30397.0
1,B,109629.0
2,C,1584.0
3,D,428047.0
4,E,368627.0
Total,NaN,938284.0


In [40]:
tabla_revision = (
    gdf_manzanas.groupby(['combo_signature', 'CVE_EQ_C'])
      .size()
      .reset_index(name='n')
      .sort_values('combo_signature')
)
tabla_revision

,combo_signature,CVE_EQ_C,n
0,S0_C0_D4,E,18708
1,S0_C1_D3,E,28409
2,S0_C2_D2,E,13808
3,S0_C3_D1,D,2952
4,S0_C4_D0,C,77
5,S1_C0_D3,E,49978
6,S1_C1_D2,E,73222
7,S1_C2_D1,D,22708
8,S1_C3_D0,C,1507
9,S2_C0_D2,E,184502


In [47]:
def asignar_clave(row):
    count_S = row['count_S']
    count_C = row['count_C']
    count_D = row['count_D']
    
    # 1️⃣ Superávit absoluto
    if count_S == 4:
        return 'A'
    
    # 2️⃣ Sin déficits (solo S y C)
    if count_D == 0:
        if (count_S == 3 and count_C == 1) or (count_S == 2 and count_C == 2):
            return 'B'
        elif count_C == 4 or (count_C == 3 and count_S == 1):
            return 'C'
    
    # 3️⃣ Déficit absoluto
    if count_D == 4:
        return 'F'
    
    # 4️⃣ Cobertura parcial (exactamente un D)
    if count_D == 1:
        return 'D'
    
    # 5️⃣ Deficiencia estructural (2 o 3 D)
    if count_D >= 2:
        return 'E'
    
    return np.nan

gdf_manzanas['CVE_EQ_C_v2'] = gdf_manzanas.apply(asignar_clave, axis=1)

In [48]:
tabla_revision = (
    gdf_manzanas.groupby(['combo_signature','CVE_EQ_C_v2'])
      .size()
      .reset_index(name='n')
      .sort_values('combo_signature')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,combo_signature,CVE_EQ_C_v2,n
0,S0_C0_D4,F,18708.0
1,S0_C1_D3,E,28409.0
2,S0_C2_D2,E,13808.0
3,S0_C3_D1,D,2952.0
4,S0_C4_D0,C,77.0
5,S1_C0_D3,E,49978.0
6,S1_C1_D2,E,73222.0
7,S1_C2_D1,D,22708.0
8,S1_C3_D0,C,1507.0
9,S2_C0_D2,E,184502.0


In [49]:
tabla_revision = (
    gdf_manzanas.groupby(['CVE_EQ_C_v2'])
      .size()
      .reset_index(name='n')
      .sort_values('CVE_EQ_C_v2')
)
tabla_revision.loc['Total'] = tabla_revision.sum(numeric_only=True)
tabla_revision

,CVE_EQ_C_v2,n
0,A,30397.0
1,B,109629.0
2,C,1584.0
3,D,428047.0
4,E,349919.0
5,F,18708.0
Total,NaN,938284.0


In [46]:
list(gdf_manzanas.columns.values)

['CVEGEO',
 'POPTOT2010',
 'POPTOT2020',
 'VIVTOT2010',
 'VIVTOT2020',
 'TVIVHAB2010',
 'TVIVHAB2020',
 'TVIVPAR2010',
 'TVIVPAR2020',
 'TVIVPARHAB2010',
 'TVIVPARHAB2020',
 'VIVPAR_HAB2010',
 'VIVPAR_HAB2020',
 'VIVPAR_UT2010',
 'VIVPAR_UT2020',
 'OCUPVIVPAR2010',
 'OCUPVIVPAR2020',
 'VIVPAR_DES2010',
 'VIVPAR_DES2020',
 'VIV1C2010',
 'VIV1C2020',
 'VIV2C2010',
 'VIV2C2020',
 'VIV3C2010',
 'VIV3C2020',
 'CVE_ENT',
 'CVE_MUN',
 'CVE_LOC',
 'CVE_AGEB',
 'CVE_MZA',
 'AMBITO',
 'TIPOMZA',
 'NOMGEO',
 'index_right',
 'CALIF_CLAS',
 'CVEMUN',
 'CVE_MET',
 'NOM_MET',
 'TIPO_MET',
 'TIPO_MUN',
 'area_ha',
 'DESHABITACION',
 'DIFPOP',
 'DIFVIV',
 'RELPOP',
 'RELVIV',
 'CVEPOP',
 'CVEVIV',
 'CVE',
 'TIPOLOGIA',
 'DENS_POP10',
 'DENS_POP20',
 'DENS_VIV10',
 'DENS_VIV20',
 'CUARTOS2010',
 'CUARTOS2020',
 'CPERC2010',
 'CPERC2020',
 'HABXVIV2010',
 'HABXVIV2020',
 'R_DESHAB',
 'SUBTIPOLOGÍA',
 'UE_10',
 'Emp_10',
 'UE_20',
 'Emp_20',
 'TAMAÑO',
 'TASA_EMP_MUN',
 'DIF_EMP',
 'DIF_UEC',
 'REL_EMP',


In [ ]:
manzanas_gpkg_path = r"D:\AFC\SSD\VOCES\DATA\Tratados\NAC\INT_MZN_NAC\Mzn_Tipologías_NAC_ImpEq_BufferB_rev.gpkg"
capa = 'Mzn_Subtipologias_UeEQ_filtrado_rev1'

#Guardar la capa resultante en un archivo .gpkg
gdf_manzanas.to_file(manzanas_gpkg_path, layer=capa, driver="GPKG")